# ChineseEEG Reading vs Non-Reading Dataset Inspection

This notebook provides quick utilities to inspect the HDF5 outputs from
`chineseeeg_reading.py`, including:

- Listing subjects
- Inspecting channel configuration
- Checking label distributions
- Visualizing example reading vs non-reading segments


In [ ]:
import sys
from pathlib import Path
import numpy as np

sys.path.insert(0, '.')

from hdf5_io import HDF5Reader


In [ ]:
# Change this to your processed dataset path
data_dir = "/mnt/dataset2/Processed_datasets/EEG_Bench/ChineseEEG_reading_binary"
data_path = Path(data_dir)
h5_files = sorted(data_path.glob("sub-*.h5"))
print(f"Found {len(h5_files)} HDF5 files in {data_dir}")


In [ ]:
# Inspect subject-level metadata for the first file
if h5_files:
    with HDF5Reader(str(h5_files[0])) as reader:
        subj = reader.subject_attrs
        print(f"Dataset name: {subj.dataset_name}")
        print(f"Task type: {subj.task_type}")
        print(f"Downstream task: {subj.downstream_task_type}")
        print(f"Subject ID: {subj.subject_id}")
        print(f"Sampling rate: {subj.rsFreq} Hz")
        print(f"Montage: {subj.montage}")
        print(f"Channels ({len(subj.chn_name)}): {subj.chn_name[:10]}...")
        print(f"Num labels: {getattr(subj, 'num_labels', 'N/A')}")
        print(f"Categories: {getattr(subj, 'category_list', 'N/A')}")
        print(f"Num trials: {len(reader.get_trial_names())}")
        print(f"Num segments: {len(reader)}")


In [ ]:
# Aggregate label statistics across all subjects
from collections import Counter

label_counts = Counter()
subject_label_counts = {}

for h5_file in h5_files:
    with HDF5Reader(str(h5_file)) as reader:
        subj = reader.subject_attrs
        subj_counts = Counter()
        for trial_name in reader.get_trial_names():
            for seg_name in reader.get_segment_names(trial_name):
                seg = reader.get_segment(trial_name, seg_name)
                lab = int(seg.segment.label[0]) if len(seg.segment.label) > 0 else None
                if lab is not None:
                    label_counts[lab] += 1
                    subj_counts[lab] += 1
        subject_label_counts[subj.subject_id] = subj_counts

print("Global label counts:")
for k, v in sorted(label_counts.items()):
    print(f"  Label {k}: {v}")


In [ ]:
# Visualize a reading vs non-reading segment
try:
    import matplotlib.pyplot as plt

    def plot_example_segment(file_path: Path, label: int = 1, max_channels: int = 8):
        with HDF5Reader(str(file_path)) as reader:
            subj = reader.subject_attrs
            sfreq = subj.rsFreq
            ch_names = subj.chn_name

            for trial_name in reader.get_trial_names():
                for seg_name in reader.get_segment_names(trial_name):
                    seg = reader.get_segment(trial_name, seg_name)
                    lab = int(seg.segment.label[0]) if len(seg.segment.label) > 0 else None
                    if lab == label:
                        data = seg.data  # (C, T)
                        n_ch, n_samp = data.shape
                        t = np.arange(n_samp) / sfreq
                        n_plot = min(max_channels, n_ch)
                        offset = np.abs(data).max() * 1.5
                        offsets = np.arange(n_plot) * offset

                        fig, ax = plt.subplots(figsize=(12, 6))
                        for i in range(n_plot):
                            ax.plot(t, data[i] + offsets[i], lw=0.7)
                            ax.text(t[-1] + 0.01 * (t[-1] - t[0]), offsets[i], ch_names[i],
                                    va='center', fontsize=8)

                        lab_name = subj.category_list[lab] if subj.category_list and lab < len(subj.category_list) else str(lab)
                        ax.set_title(f"Subject {subj.subject_id} - Example segment (label={lab_name})")
                        ax.set_xlabel("Time (s)")
                        ax.set_ylabel("Amplitude (µV, offset by channel)")
                        ax.grid(True, alpha=0.3)
                        plt.tight_layout()
                        plt.show()
                        return

            print(f"No segment with label {label} found in {file_path.name}")

    if h5_files:
        print("\nPlotting example READING segment (label=1) from first subject...")
        plot_example_segment(h5_files[0], label=1)
        print("\nPlotting example NON-READING segment (label=0) from first subject...")
        plot_example_segment(h5_files[0], label=0)
except ImportError:
    print("matplotlib not available, skipping plots")
